# Notebook 3: Package the ranker with BentoML for MLIS deployment

Wrap the trained XGBoost model in a BentoML service exposing
**prediction and SHAP-style explanation only** — no pipe-table lookup, no
business logic. Data lives in EzPresto; the model server does model math.

**Outputs**

- A **Bento** in the local BentoML store (name: `water-utility-ranker`)
- A **container image** produced by `bentoml containerize`, tagged for Docker Hub
- A push command block ready to copy
- MLIS registration steps (which endpoint URL to expect back)

**What this notebook DOES NOT do**

- Publish the image (that's a `docker push` from a terminal with Docker Hub credentials)
- Deploy to MLIS (that's a UI step — instructions at the bottom)
- Query EzPresto (that's the OWU tool's job at demo time)

In [5]:
import os
import shutil
import subprocess
from pathlib import Path

SHARED_DIR = Path(os.environ.get("SHARED_DIR", "shared"))
MODEL_PATH = SHARED_DIR / "ranker_v1.json"
BENTO_STAGE_DIR = SHARED_DIR / "bento-stage"

BENTO_SERVICE_NAME = "water_utility_ranker"
BENTO_TAG = "water_utility_ranker:0.2.0"
IMAGE_REPO = os.environ.get("IMAGE_REPO", "caovd/water-utility-ranker")
IMAGE_TAG = os.environ.get("IMAGE_TAG", "0.2.0")

assert MODEL_PATH.exists(), f"model artifact missing at {MODEL_PATH} — run notebook 01 first"

## 1. Import the trained model into the BentoML model store

BentoML's model store lets us reference the trained artifact by name inside the
service code. We register `water_utility_ranker:latest` pointing at
`shared/ranker_v1.json`.

In [6]:
import bentoml
import xgboost as xgb

booster = xgb.Booster()
booster.load_model(str(MODEL_PATH))

# Save as an XGBoost Booster (not sklearn wrapper — the service loads the Booster directly)
bento_model = bentoml.xgboost.save_model(
    "water_utility_ranker",
    booster,
    metadata={
        "framework": "xgboost",
        "trained_by": "notebook_01_train_with_mlflow",
        "feature_names": [
            "age_years", "material_ordinal", "diameter_mm", "slope_pct", "depth_m",
            "nearest_tree_dist_m", "trees_within_15m", "dominant_species_riskscore",
            "historical_incident_count",
        ],
    },
)
print(f"registered in BentoML model store: {bento_model.tag}")

Using the default model signature for xgboost ({'predict': {'batchable': False}}) for model "water_utility_ranker".
HTTP Request: POST https://t.bentoml.com "HTTP/1.1 200 OK"


registered in BentoML model store: water_utility_ranker:rwb4jvuum6lvh7we


## 2. Stage the Bento service source

We write the three source files (`service.py`, `bentofile.yaml`, `requirements.txt`)
directly to the staging directory. This keeps the notebook self-contained — you can
run it without uploading a separate `bento/` folder alongside.

If you later want to modify the service (e.g. add a new endpoint), edit the strings
in the cells below and re-run.

In [7]:
if BENTO_STAGE_DIR.exists():
    shutil.rmtree(BENTO_STAGE_DIR)
BENTO_STAGE_DIR.mkdir(parents=True)

SERVICE_PY = '''\
"""
BentoML service — Water Utility Tree Root Risk Ranker.

Pure inference: /predict + /explain. No data lookup — pipe rows come from EzPresto.

BentoML v1.4+ class-decorator API. No Runners; the model loads directly in __init__.
"""
from __future__ import annotations
from typing import List

import bentoml
import numpy as np
import xgboost as xgb
from pydantic import BaseModel, Field


FEATURES = [
    "age_years", "material_ordinal", "diameter_mm", "slope_pct", "depth_m",
    "nearest_tree_dist_m", "trees_within_15m", "dominant_species_riskscore",
    "historical_incident_count",
]


class FeatureRow(BaseModel):
    age_years: float
    material_ordinal: int = Field(..., ge=0, le=4)
    diameter_mm: float
    slope_pct: float
    depth_m: float
    nearest_tree_dist_m: float
    trees_within_15m: int = Field(..., ge=0)
    dominant_species_riskscore: float = Field(..., ge=0.0, le=1.0)
    historical_incident_count: int = Field(..., ge=0)


@bentoml.service(
    name="water_utility_ranker",
    resources={"cpu": "1"},
    traffic={"timeout": 30},
)
class WaterUtilityRanker:
    def __init__(self):
        # Loads once per replica startup from the BentoML model store
        self.booster = bentoml.xgboost.load_model("water_utility_ranker:latest")

    def _to_dmatrix(self, rows: List[FeatureRow]) -> xgb.DMatrix:
        arr = np.array(
            [[getattr(r, f) for f in FEATURES] for r in rows],
            dtype=float,
        )
        return xgb.DMatrix(arr, feature_names=FEATURES)

    @bentoml.api(route="/predict")
    async def predict(self, instances: List[FeatureRow]) -> dict:
        """Batch prediction — one probability per feature row."""
        if not instances:
            return {"predictions": []}
        dmat = self._to_dmatrix(instances)
        proba = self.booster.predict(dmat)
        return {"predictions": [float(p) for p in np.atleast_1d(proba)]}

    @bentoml.api(route="/explain")
    async def explain(self, row: FeatureRow) -> dict:
        """SHAP-style per-feature contributions for one row."""
        dmat = self._to_dmatrix([row])
        contribs = self.booster.predict(dmat, pred_contribs=True)
        contribs = np.atleast_2d(contribs)[0]
        bias = float(contribs[-1])
        feat_contribs = {FEATURES[i]: float(contribs[i]) for i in range(len(FEATURES))}
        risk_score = float(1.0 / (1.0 + np.exp(-float(contribs.sum()))))
        top = sorted(feat_contribs.items(), key=lambda kv: abs(kv[1]), reverse=True)[:5]
        top_items = [
            {
                "feature": name,
                "value": float(getattr(row, name)),
                "contribution": round(val, 4),
                "direction": "increases risk" if val > 0 else "decreases risk",
            }
            for name, val in top
        ]
        return {
            "risk_score": round(risk_score, 4),
            "bias": round(bias, 4),
            "top_contributions": top_items,
        }

    @bentoml.api(route="/healthz")
    async def healthz(self) -> dict:
        return {"status": "ok", "features": FEATURES}
'''

BENTOFILE_YAML = """\
service: "service:WaterUtilityRanker"
labels:
  owner: daniel-cao
  use_case: tree-root-infiltration-risk
  version: "0.2.0"
include:
  - "service.py"
python:
  requirements_txt: "./requirements.txt"
  lock_packages: false
docker:
  distro: debian
  python_version: "3.11"
  system_packages: []
models:
  - "water_utility_ranker:latest"
"""

REQUIREMENTS_TXT = """\
bentoml>=1.4.0,<2.0.0
xgboost==2.1.2
numpy>=1.26,<3.0
pydantic>=2.9.0,<3.0.0
"""

(BENTO_STAGE_DIR / "service.py").write_text(SERVICE_PY)
(BENTO_STAGE_DIR / "bentofile.yaml").write_text(BENTOFILE_YAML)
(BENTO_STAGE_DIR / "requirements.txt").write_text(REQUIREMENTS_TXT)

print("staged:")
for p in sorted(BENTO_STAGE_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size} bytes)")

staged:
  bentofile.yaml  (337 bytes)
  requirements.txt  (77 bytes)
  service.py  (3087 bytes)


## 3. Build the Bento

`bentoml build` reads `bentofile.yaml`, snapshots the service source and the
referenced model, and produces a versioned Bento in `~/bentoml/bentos/`.

In [8]:
build = subprocess.run(
    ["bentoml", "build", str(BENTO_STAGE_DIR)],
    capture_output=True, text=True, check=False,
)
print(build.stdout)
if build.returncode != 0:
    print("STDERR:", build.stderr)
    raise RuntimeError("bentoml build failed")

# List bentos for this service to confirm
subprocess.run(["bentoml", "list", "water_utility_ranker"], check=False)

INFO: uv is not installed, installing it with: /opt/conda/bin/python3.11 -m pip install -q uv
INFO: Adding BentoML requirement to the image: bentoml==1.4.39.

██████╗ ███████╗███╗   ██╗████████╗ ██████╗ ███╗   ███╗██╗
██╔══██╗██╔════╝████╗  ██║╚══██╔══╝██╔═══██╗████╗ ████║██║
██████╔╝█████╗  ██╔██╗ ██║   ██║   ██║   ██║██╔████╔██║██║
██╔══██╗██╔══╝  ██║╚██╗██║   ██║   ██║   ██║██║╚██╔╝██║██║
██████╔╝███████╗██║ ╚████║   ██║   ╚██████╔╝██║ ╚═╝ ██║███████╗
╚═════╝ ╚══════╝╚═╝  ╚═══╝   ╚═╝    ╚═════╝ ╚═╝     ╚═╝╚══════╝

Successfully built Bento(tag="water_utility_ranker:tvo2oruum6naf7we").

Next steps: 

* Deploy to BentoCloud:
    $ bentoml deploy water_utility_ranker:tvo2oruum6naf7we -n ${DEPLOYMENT_NAME}

* Update an existing deployment on BentoCloud:
    $ bentoml deployment update --bento water_utility_ranker:tvo2oruum6naf7we 
${DEPLOYMENT_NAME}

* Containerize your Bento with `bentoml containerize`:
    $ bentoml containerize water_utility_ranker:tvo2oruum6naf7we 

* Push to BentoC

CompletedProcess(args=['bentoml', 'list', 'water_utility_ranker'], returncode=0)

## 4. Containerize

`bentoml containerize` invokes Docker (or nerdctl / kaniko if configured) to build
an OCI image from the Bento. On AIE Jupyter kernels this usually shells out to the
Docker CLI — verify Docker is available first.

In [9]:
docker_check = subprocess.run(["which", "docker"], capture_output=True, text=True)
if docker_check.returncode != 0:
    print(
        "\n[warn] `docker` not found in this Jupyter kernel. "
        "Run the containerize + push steps from a workstation with Docker installed. "
        "Copy the Bento first with:\n"
        f"    bentoml export {BENTO_TAG} ./water_utility_ranker.bento\n"
        "then on your workstation:\n"
        f"    bentoml import water_utility_ranker.bento\n"
        f"    bentoml containerize {BENTO_TAG} -t {IMAGE_REPO}:{IMAGE_TAG}\n"
    )
else:
    containerize = subprocess.run(
        ["bentoml", "containerize", BENTO_TAG, "-t", f"{IMAGE_REPO}:{IMAGE_TAG}"],
        capture_output=True, text=True, check=False,
    )
    print(containerize.stdout[-2000:])
    if containerize.returncode != 0:
        print("STDERR:", containerize.stderr[-2000:])
        raise RuntimeError("bentoml containerize failed")


[warn] `docker` not found in this Jupyter kernel. Run the containerize + push steps from a workstation with Docker installed. Copy the Bento first with:
    bentoml export water_utility_ranker:0.2.0 ./water_utility_ranker.bento
then on your workstation:
    bentoml import water_utility_ranker.bento
    bentoml containerize water_utility_ranker:0.2.0 -t caovd/water-utility-ranker:0.2.0



In [13]:
# bentoml import "$PWD/water_utility_ranker.bento"

In [12]:
!bentoml export water_utility_ranker:latest "$PWD/water_utility_ranker.bento"

Bento(tag="water_utility_ranker:tvo2oruum6naf7we") exported to 
/mnt/user/water-utility-ranker/water_utility_ranker.bento.


## 5. Push to Docker Hub

**Do not run inside this notebook** — perform from a terminal that has Docker Hub
credentials configured. Copy-paste the block below:

In [ ]:
push_commands = f"""
# From a terminal with Docker Hub credentials:
docker login -u caovd
docker push {IMAGE_REPO}:{IMAGE_TAG}

# Verify:
docker manifest inspect {IMAGE_REPO}:{IMAGE_TAG} | head -15
""".strip()
print(push_commands)

## 6. Register in MLIS UI (Gate 2 path: custom container)

1. Open **AIE UI → MLIS**.
2. Click **Deploy Model** → **Custom Container**.
3. Fill the form:

   | Field | Value |
   |---|---|
   | Model name | `water-utility-ranker` |
   | Image URI | `docker.io/{IMAGE_REPO}:{IMAGE_TAG}` |
   | Container port | `3000` (BentoML default) |
   | Health check path | `/healthz` |
   | Endpoint schema | OpenAPI (BentoML exposes `/docs`) |
   | Replicas | `1` |
   | GPU | `0` (CPU model, integer allocation would waste an H200) |
   | Resources | CPU `500m` request / `2000m` limit, RAM `1Gi` / `2Gi` |

4. Deploy. MLIS returns an endpoint URL like:

   `https://water-utility-ranker.<mlis-domain>/`

5. Sanity-check from a terminal (substitute the URL and your MLIS API key):

   ```bash
   MLIS_URL="https://water-utility-ranker.<mlis-domain>"
   KEY="<mlis-api-key>"

   curl -sfH "Authorization: Bearer $KEY" "$MLIS_URL/healthz"

   curl -sfH "Authorization: Bearer $KEY" -H 'Content-Type: application/json' \\
     -X POST "$MLIS_URL/predict" \\
     -d '{{"instances":[{{"age_years":63,"material_ordinal":4,"diameter_mm":300,
          "slope_pct":1.2,"depth_m":2.1,"nearest_tree_dist_m":4.0,
          "trees_within_15m":3,"dominant_species_riskscore":0.9,
          "historical_incident_count":3}}]}}'
   ```

You should see `{"predictions": [0.9x]}` back. That URL is what you paste into the
OWU tool valves.

## Pass criteria

| Criterion | Value |
|---|---|
| Bento built with tag `water_utility_ranker:0.2.0` | printed above |
| Container image tagged for `caovd/water-utility-ranker:0.2.0` | printed above |
| Push commands surfaced (not executed) | ✅ |
| MLIS registration steps documented | ✅ |

**Next**: register EzPresto catalogs (see `ezpresto/EZPRESTO_SETUP.md`), then upload
the OWU Tools file (`owu-tools/water_utility_ranker.py`) and run the demo beats.